# 🛡️ Network Intrusion Detection System (NIDS) — NSL-KDD

A binary network intrusion classifier built on the **NSL-KDD** benchmark dataset, comparing
**Logistic Regression**, **Decision Tree**, and **Random Forest** models — with threshold
tuning to favor catching real attacks (recall) over raw accuracy.

> 📄 See [`docs/NIDS_Code_Explanation.pdf`](docs/NIDS_Code_Explanation.pdf) or the companion
> markdown walkthrough for a full line-by-line explanation of this notebook.

---


## 1️⃣ Imports & Setup

Load the core data-science stack (NumPy, Pandas, Matplotlib, Seaborn) and the scikit-learn
tools used for preprocessing, model training, and evaluation.

In [ ]:
"""
===========================================
  Network Intrusion Detector
  Dataset : NSL-KDD (real Kaggle dataset)
  Models  : Logistic Regression, Decision Tree, Random Forest
===========================================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve
)

## 2️⃣ Loading the Dataset

The **NSL-KDD** dataset has no header row, so we manually declare all 42 column names, then
load the train and test CSVs into Pandas DataFrames.

> ⚠️ Update `TRAIN_PATH` and `TEST_PATH` below to point at your local copy of
> `KDDTrain.csv` / `KDDTest.csv` (see [Setup](#setup) in the README for the download link).

In [ ]:
COLUMNS = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised',
    'root_shell','su_attempted','num_root','num_file_creations','num_shells',
    'num_access_files','num_outbound_cmds','is_host_login','is_guest_login',
    'count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
    'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count',
    'dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
    'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'label','difficulty'
]

print("=" * 57)
print("   NETWORK INTRUSION DETECTION SYSTEM — NSL-KDD")
print("=" * 57)

TRAIN_PATH = "data/KDDTrain.csv"
TEST_PATH  = "data/KDDTest.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"\n📦 Train: {train_df.shape[0]:,} rows | Test: {test_df.shape[0]:,} rows")

## 3️⃣ Binary Labeling

NSL-KDD ships with dozens of specific attack types (`neptune`, `smurf`, `portsweep`, ...).
We collapse these into a **binary** problem — `0 = Normal`, `1 = Attack` — since a real-time
IDS mainly needs to answer *"is this suspicious?"*

In [ ]:
train_df['binary_label'] = (train_df['label'] != 'normal').astype(int)
test_df['binary_label']  = (test_df['label']  != 'normal').astype(int)

print(f"\n📊 Train class distribution:")
vc = train_df['binary_label'].map({0:'Normal',1:'Attack'}).value_counts()
for k, v in vc.items():
    print(f"   {k}: {v:,}  ({v/len(train_df)*100:.1f}%)")

## 4️⃣ Preprocessing

Converts every categorical column to numeric form and scales features:
- **Label encoding** for low-cardinality columns (`protocol_type`, `flag`)
- **One-hot encoding** for the high-cardinality `service` column (70+ values, no natural order)
- **StandardScaler** for the features that Logistic Regression needs on a common scale

In [ ]:
print("\n🔧 Preprocessing...")

# protocol_type and flag have few categories (3 and 11) -> label encoding is fine for trees
small_cat_cols = ['protocol_type', 'flag']
le = LabelEncoder()
for col in small_cat_cols:
    combined = pd.concat([train_df[col], test_df[col]])
    le.fit(combined)
    train_df[col] = le.transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])

# 'service' has 70+ categories with NO natural order (http, ftp, smtp...).
# Label-encoding it as 1,2,3... would falsely imply ranking, so we one-hot encode instead.
combined_service = pd.concat([train_df['service'], test_df['service']])
service_dummies_all = pd.get_dummies(combined_service, prefix='service')
n_train = len(train_df)
train_service_dummies = service_dummies_all.iloc[:n_train].reset_index(drop=True)
test_service_dummies  = service_dummies_all.iloc[n_train:].reset_index(drop=True)

train_df = pd.concat([train_df.drop(columns=['service']).reset_index(drop=True), train_service_dummies], axis=1)
test_df  = pd.concat([test_df.drop(columns=['service']).reset_index(drop=True),  test_service_dummies],  axis=1)

FEATURE_COLS = [c for c in train_df.columns if c not in ('label', 'difficulty', 'binary_label')]

X_train = train_df[FEATURE_COLS].values
y_train = train_df['binary_label'].values
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df['binary_label'].values

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"   Features : {X_train.shape[1]}")
print(f"   Train    : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

## 5️⃣ Training the Models

Three classifiers are trained in a loop — **Logistic Regression**, **Decision Tree**, and
**Random Forest** — each evaluated with accuracy, ROC-AUC, and 5-fold cross-validation.

In [ ]:
print("\n🚀 Training models...\n")

models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=42), True),
    "Decision Tree":       (DecisionTreeClassifier(max_depth=15, random_state=42), False),
    "Random Forest":       (RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42), False),
}

results = {}

for name, (model, use_scaled) in models.items():
    Xtr = X_train_sc if use_scaled else X_train
    Xte = X_test_sc  if use_scaled else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    cv  = cross_val_score(model, Xtr, y_train, cv=5, scoring='accuracy')

    results[name] = dict(model=model, y_pred=y_pred, y_prob=y_prob,
                         accuracy=acc, auc=auc, cv_mean=cv.mean(),
                         cv_std=cv.std(), Xte=Xte)

    print(f"✅ {name}")
    print(f"   Accuracy : {acc:.4f}")
    print(f"   ROC-AUC  : {auc:.4f}")
    print(f"   CV Acc   : {cv.mean():.4f} ± {cv.std():.4f}")
    print()

## 6️⃣ Threshold Tuning

By default a model predicts "Attack" only above 50% confidence. In security, a missed
attack is far costlier than a false alarm, so we lower the decision threshold to **0.35**
to trade a little precision for a meaningful boost in attack recall.

In [ ]:
# Default cutoff is 0.5 ("Attack" only if model is >50% sure).
# In intrusion detection, missing a real attack is worse than a false alarm,
# so we lower the cutoff to 0.35 to catch more attacks, at the cost of a few more false alarms.
THRESHOLD = 0.35

best_so_far = max(results, key=lambda k: results[k]['auc'])
y_prob_best = results[best_so_far]['y_prob']
y_pred_tuned = (y_prob_best >= THRESHOLD).astype(int)

print("=" * 57)
print(f"  THRESHOLD TUNING ON BEST MODEL: {best_so_far}")
print("=" * 57)
print(f"\nDefault threshold (0.5) Attack recall : {results[best_so_far]['y_pred'][y_test==1].mean():.3f}")
print(f"Tuned threshold ({THRESHOLD}) Attack recall : {y_pred_tuned[y_test==1].mean():.3f}")
print(f"\nNew accuracy at threshold {THRESHOLD}: {accuracy_score(y_test, y_pred_tuned):.4f}")
print("\nNew Classification Report (tuned threshold):")
print(classification_report(y_test, y_pred_tuned, target_names=['Normal', 'Attack']))

# Save tuned predictions so later cells (Section 7 demo) can use them
results[best_so_far]['y_pred_tuned'] = y_pred_tuned

## 7️⃣ Best-Model Report

Picks the model with the highest ROC-AUC and prints its full precision/recall/F1 breakdown.

In [ ]:
best_name = max(results, key=lambda k: results[k]['auc'])
print(f"🏆 Best Model: {best_name}")
print("\n📋 Classification Report:")
print(classification_report(y_test, results[best_name]['y_pred'],
                             target_names=['Normal', 'Attack']))

## 8️⃣ Evaluation & Visualizations

A 2×3 dashboard: confusion matrices for all three models (top row), plus an accuracy/AUC
comparison, ROC curves, and Random Forest feature importances (bottom row).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Network Intrusion Detector — NSL-KDD Results", fontsize=16, fontweight='bold')

colors = ["#4C72B0", "#DD8452", "#55A868"]
short  = ["Log. Reg.", "Dec. Tree", "Rand. Forest"]

# Row 1 — Confusion Matrices
for ax, (name, res), col in zip(axes[0], results.items(), colors):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal','Attack'],
                yticklabels=['Normal','Attack'], ax=ax)
    ax.set_title(f"{name}\nAcc: {res['accuracy']:.3f}", fontsize=11)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

# Row 2, Col 1 — Accuracy & AUC bars
ax = axes[1][0]
x = np.arange(3); w = 0.35
accs = [results[k]['accuracy'] for k in results]
aucs = [results[k]['auc']      for k in results]
b1 = ax.bar(x-w/2, accs, w, label='Accuracy', color=colors, alpha=0.85)
b2 = ax.bar(x+w/2, aucs, w, label='ROC-AUC',  color=colors, alpha=0.5, hatch='//')
ax.set_xticks(x); ax.set_xticklabels(short, fontsize=9)
ax.set_ylim(0.85, 1.01); ax.set_title("Accuracy vs ROC-AUC"); ax.legend(fontsize=8)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.002,
            f"{b.get_height():.3f}", ha='center', va='bottom', fontsize=8)

# Row 2, Col 2 — ROC Curves
ax = axes[1][1]
for (name, res), col, s in zip(results.items(), colors, short):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=col, lw=2, label=f"{s} (AUC={res['auc']:.3f})")
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves"); ax.legend(fontsize=8)

# Row 2, Col 3 — Feature Importance (RF)
ax = axes[1][2]
rf = results['Random Forest']['model']
imp = rf.feature_importances_
top = np.argsort(imp)[-15:]
ax.barh([FEATURE_COLS[i] for i in top], imp[top], color='#55A868', alpha=0.85)
ax.set_title("Top 15 Features\n(Random Forest)"); ax.set_xlabel("Importance")
ax.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.show()

## 9️⃣ Live Prediction Demo

Randomly samples 5 test rows and walks through the tuned-threshold prediction for each —
actual label, predicted label, model confidence, and correctness.

In [ ]:
print("\n" + "=" * 57)
print("  🔍 LIVE PREDICTION DEMO (5 random test samples)")
print("=" * 57)

np.random.seed(42)  # so the demo is reproducible every run
rf_model = results['Random Forest']['model']
idxs = np.random.choice(len(X_test), 5, replace=False)
print(f"\n{'#':<6} {'Actual':<10} {'Predicted':<12} {'Confidence':<12} {'Result'}")
print("-" * 55)
for i in idxs:
    sample = X_test[i].reshape(1, -1)
    actual = 'Normal' if y_test[i] == 0 else 'Attack'
    prob_attack = rf_model.predict_proba(sample)[0][1]
    pred   = 'Attack' if prob_attack >= THRESHOLD else 'Normal'  # use tuned threshold
    conf   = prob_attack if pred == 'Attack' else 1 - prob_attack
    status = "✅" if actual == pred else "❌"
    print(f"{i:<6} {actual:<10} {pred:<12} {conf:<12.2%} {status}")

print(f"\n✅ Done!  Best model → {best_name}  |  AUC: {results[best_name]['auc']:.4f}")

## 🔟 Mixed-Confidence Demo

An improved version of the live demo above: instead of picking 5 arbitrary rows, it
deliberately samples rows where the model's confidence is between **55–90%** — showing
the model genuinely weighing evidence rather than making trivially easy calls.

### Why test accuracy is ~77% while cross-validation accuracy is ~99%

This gap is **expected, not a bug**. `KDDTest.csv` intentionally includes attack types that
never appear in `KDDTrain.csv`, simulating real-world zero-day attacks that no model can
recognize on first sight. Published NSL-KDD binary-classification benchmarks typically fall
in the 75–82% test-accuracy range — a figure near 99% would actually be a red flag for data
leakage. Threshold tuning above is the mitigation applied: trading a few extra false alarms
for meaningfully better attack recall.

In [ ]:
# ─────────────────────────────────────────────
# 7b. LIVE PREDICTION DEMO — MIXED CONFIDENCE
# ─────────────────────────────────────────────
# The demo above can randomly land on 5 "easy" rows where the model is 100% sure.
# This version deliberately picks rows where the model is LESS than fully certain
# (confidence between 55% and 90%), so the demo shows the model actually weighing
# evidence rather than five trivially easy guesses.

print("\n" + "=" * 57)
print("  🔍 LIVE PREDICTION DEMO (5 mixed-confidence samples)")
print("=" * 57)

rf_model = results['Random Forest']['model']
all_probs = rf_model.predict_proba(X_test)[:, 1]  # probability of "Attack" for every test row

# confidence = how sure the model is about whichever class it picked
confidence_per_row = np.where(all_probs >= 0.5, all_probs, 1 - all_probs)

# keep only rows where the model is reasonably but not perfectly sure
mixed_mask = (confidence_per_row >= 0.55) & (confidence_per_row <= 0.90)
mixed_idxs = np.where(mixed_mask)[0]

print(f"\n({len(mixed_idxs):,} of {len(X_test):,} test rows fall in the 55-90% confidence range)")

np.random.seed(7)
if len(mixed_idxs) >= 5:
    chosen = np.random.choice(mixed_idxs, 5, replace=False)
else:
    chosen = mixed_idxs  # fallback if very few rows qualify

print(f"\n{'#':<6} {'Actual':<10} {'Predicted':<12} {'Confidence':<12} {'Result'}")
print("-" * 55)
for i in chosen:
    sample = X_test[i].reshape(1, -1)
    actual = 'Normal' if y_test[i] == 0 else 'Attack'
    prob_attack = rf_model.predict_proba(sample)[0][1]
    pred   = 'Attack' if prob_attack >= THRESHOLD else 'Normal'  # tuned threshold
    conf   = prob_attack if pred == 'Attack' else 1 - prob_attack
    status = "✅" if actual == pred else "❌"
    print(f"{i:<6} {actual:<10} {pred:<12} {conf:<12.2%} {status}")